# Probabilistic matching: Fellegi–Sunter on what deterministic matching left

Builds on the deterministic stage's files in `data/artifacts/*/<dataset>/deterministic_*`. Per dataset:

1. **Candidates (blocking):** pairs worth scoring, for every record the deterministic stage left unresolved, against **all** records. Generators:
   - exact keys
   - sorted neighbourhood
   - TF-IDF character n-grams

   Each is measured on its own and cumulatively.
2. **Matching (scoring):** Splink's **Fellegi–Sunter** model scores exactly those candidate pairs, so any generator can be combined with any scorer.
   - prior λ from near-certain rules
   - u by random sampling
   - m by **EM** (unsupervised) or from **labels**

   Comparison levels use the character-level `cl_*` functions; correlated columns share one comparison.
3. **Decision:** accept at `match_weight ≥ min_match_weight`. The weight is chosen by a `ThresholdRule` on **out-of-fold train scores**: entity-level k-fold cross-validation, so the model never scores records it was fitted on.
4. **Resolution:** deterministic pairs + accepted pairs → the configured `Resolver`.
5. **Evaluation:** with uncertainty:
   - `recall_lo` / `recall_hi`: 95% Wilson interval
   - `fp_rate_hi`: 95% Clopper–Pearson upper bound on the FP rate; zero FP among n only bounds the rate at ≈ 3/n

**How each dataset is used (against overfitting):**
- **train:** fitting, design analysis, the threshold (cross-validated), Optuna (cross-validated objective)
- **val:** A/B tests (paired against `SPEC`) and confirming the Pareto choice
- **test:** the final report, once per configuration (the leaderboard counts test evaluations)
- **stress / large:** robustness and FP certification, report-only

A variant replaces the default only by a stated rule (Decision cell): a paired win on cross-validated train, and no loss on the robustness datasets.

**Outputs**, per dataset:
- `data/artifacts/{matched,resolved,unresolved}/<dataset>/probabilistic_*`
- `data/medallion/gold/<dataset>.csv`: the resolved entities

In [ ]:
import json
import logging
import time
from dataclasses import replace
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from splink import Linker

from er.candidates import estimate
from er.config import (Budget, Comparator, Comparison, Data, Derived, FSModel, Key, KeyBlocker, Level, OverBudget, ProbSpec,
                       ProbTune, Resolver, SortedNeighbourhood, Standardization, TfidfNeighbours, ThresholdRule, keys_sql,
                       with_level, without_tf)
from er.evaluate import c2, candidate_metrics, missed_pairs, paired_test, predicted_pairs, true_pairs
from er.io import append_leaderboard, load, read_stage, test_evaluations, write_gold, write_stage
from er.match import parameters, score_fs, train_fs, u_values
from er.pipeline import (ProbResult, cross_validate_fs, cv_threshold_curve, entity_folds, run_fs, select_threshold, summed,
                         threshold_curve)
from er.standardize import prepare, self_check
from er.strings import connect

## Configuration
The only cell that names columns. The candidate keys are justified by the train-only design analysis below. The comparisons group correlated columns, e.g. SOURCE_ADDR_CD / ADDRESS / POSTAL form one *address* comparison.

In [ ]:
DATA = Data(
    silver_dir=Path("../data/medallion/silver"),
    labels_dir=Path("../data/labels"),
    gold_dir=Path("../data/medallion/gold"),
    artifacts_dir=Path("../data/artifacts"),
    experiments_dir=Path("../data/artifacts/experiments"),
    models_dir=Path("../data/artifacts/models"),
    datasets=("full", "train", "val", "test", "stress", "large"),
    robustness=("stress", "large"),
    unique_id="CONT_ID",
    entity_id="ENTITY_ID",
    show_rows=8,
)

STANDARDIZE = Standardization(
    kana_fold=dict(zip("ガギグゲゴザジズゼゾダヂヅデドバビブベボパピプペポヴァィゥェォッャュョヮヵヶヲー",
                       "カキクケコサシスセソタチツテトハヒフヘホハヒフヘホフアイウエオツヤユヨワカケオウ", strict=True)),
    legal_forms=("カブシキガイシャ", "ユウゲンガイシャ", "ゴウドウガイシャ", "(カ)", "(ユ)"),
    prefectures=tuple("北海道 青森県 岩手県 宮城県 秋田県 山形県 福島県 茨城県 栃木県 群馬県 埼玉県 千葉県 東京都 神奈川県 新潟県 富山県 "
                      "石川県 福井県 山梨県 長野県 岐阜県 静岡県 愛知県 三重県 滋賀県 京都府 大阪府 兵庫県 奈良県 和歌山県 鳥取県 島根県 "
                      "岡山県 広島県 山口県 徳島県 香川県 愛媛県 高知県 福岡県 佐賀県 長崎県 熊本県 大分県 宮崎県 鹿児島県 沖縄県".split()),
    address_replacements=((r"(\d+)丁目", r"\1-"), (r"(\d+)番地?", r"\1-"), (r"(\d+)号", r"\1")),
    address_cut_at=" ",
    derived=(
        Derived(name="PERSON_NAME", sources=(("FULL_NAME_KANA",), ("O_PERSON_NAME_KANA",), ("LAST_NAME_KANA", "GIVEN_NAME_KANA")),
                normalize=("strip_spaces",)),
        Derived(name="POSTAL", sources=(("POST_CD",), ("POST_CD_KANA",))),
        Derived(name="ADDRESS", sources=(("STD_ADDRESS",), ("ORIGINAL_ADDRESS",))),
    ),
)

FOLD = ("kana_fold",)
COMPANY = ("nfkc", "strip_legal_form", "kana_fold")

GENERATORS = (  # every candidate generator, compared in the Candidates cell
    KeyBlocker(name="phone", keys=(Key("PHONE_NUM_4"),)),
    KeyBlocker(name="address", keys=(Key("ADDRESS", ("address",)),)),
    KeyBlocker(name="address_code", keys=(Key("SOURCE_ADDR_CD"),)),
    KeyBlocker(name="cif", keys=(Key("CIF_ID"),)),
    KeyBlocker(name="person_name", keys=(Key("PERSON_NAME", FOLD),)),
    KeyBlocker(name="company_name", keys=(Key("NYS_CNV_P_NAME_KANA", COMPANY),)),
    SortedNeighbourhood(name="sn_person_name", key=Key("PERSON_NAME", FOLD), window=4),
    SortedNeighbourhood(name="sn_company_name", key=Key("NYS_CNV_P_NAME_KANA", COMPANY), window=4),
    TfidfNeighbours(name="tfidf_person_name", key=Key("PERSON_NAME", FOLD), ngram=(2, 3), k=5, min_cosine=0.5),
    TfidfNeighbours(name="tfidf_company_name", key=Key("NYS_CNV_P_NAME_KANA", COMPANY), ngram=(2, 3), k=5, min_cosine=0.5),
)

SPEC = ProbSpec(
    standardize=STANDARDIZE,
    previous_stage="deterministic",
    stage="probabilistic",
    blockers=GENERATORS,
    model=FSModel(
        name="fs",
        comparisons=(
            Comparison(name="name", null_if_missing=("PERSON_NAME",), tf=Key("PERSON_NAME", FOLD), levels=(
                Level(label="exact", any_of=(("PERSON_NAME", Comparator("exact")),), tf=True),
                Level(label="fold exact", any_of=(("PERSON_NAME", Comparator("exact", normalize=FOLD)),), tf=True),
                Level(label="fold JW high", any_of=(("PERSON_NAME", Comparator("jaro_winkler", 0.92, FOLD)),)),
                Level(label="given name", any_of=(("GIVEN_NAME_KANA", Comparator("exact", normalize=FOLD)),)),  # surname change
                Level(label="fold JW low", any_of=(("PERSON_NAME", Comparator("jaro_winkler", 0.85, FOLD)),)),
            )),
            Comparison(name="company", null_if_missing=("NYS_CNV_P_NAME_KANA",), tf=Key("NYS_CNV_P_NAME_KANA", COMPANY), levels=(
                Level(label="exact", any_of=(("NYS_CNV_P_NAME_KANA", Comparator("exact", normalize=COMPANY)),), tf=True),
                Level(label="JW high", any_of=(("NYS_CNV_P_NAME_KANA", Comparator("jaro_winkler", 0.92, COMPANY)),)),
                Level(label="JW low", any_of=(("NYS_CNV_P_NAME_KANA", Comparator("jaro_winkler", 0.85, COMPANY)),)),
            )),
            Comparison(name="address", null_if_missing=("SOURCE_ADDR_CD", "ADDRESS", "POSTAL"), levels=(
                Level(label="same address", any_of=(("SOURCE_ADDR_CD", Comparator("exact")),
                                                    ("ADDRESS", Comparator("exact", normalize=("address",))))),
                Level(label="address close", any_of=(("ADDRESS", Comparator("levenshtein", 2, ("address",))),)),
                Level(label="same postal", any_of=(("POSTAL", Comparator("exact")),)),
            )),
            Comparison(name="phone", null_if_missing=("PHONE_NUM_4",), levels=(
                Level(label="exact", any_of=(("PHONE_NUM_4", Comparator("exact")),)),
                Level(label="one digit off", any_of=(("PHONE_NUM_4", Comparator("hamming", 1)),)),
            )),
            Comparison(name="cif", null_if_missing=("CIF_ID",), levels=(
                Level(label="exact", any_of=(("CIF_ID", Comparator("exact")),)),
            )),
        ),
        prior_rules=((Key("CIF_ID"), Key("FULL_NAME_KANA")), (Key("CIF_ID"), Key("NYS_CNV_P_NAME_KANA")),
                     (Key("FULL_NAME_KANA"), Key("SOURCE_ADDR_CD"), Key("PHONE_NUM_4"))),
        prior_recall=0.5,  # those rules recall about half of the true pairs (deterministic exact stage, measured)
        u_max_pairs=1e7,
        m_method="em",
        em_rules=((Key("PHONE_NUM_4"),), (Key("ADDRESS", ("address",)),)),
        seed=0,
    ),
    min_match_weight=4.0,  # replaced by THRESHOLD_RULE on cross-validated train scores (Threshold cell)
    resolver=Resolver(method="connected_components"),
    budget=Budget(max_candidate_pairs=5_000_000, max_block_size=5_000, target_rows=1_000_000, benchmark_pairs=20_000),
)

CV_FOLDS, CV_SEED = 3, 0
THRESHOLD_RULE = ThresholdRule(grid=(0, 2, 4, 6, 8, 10, 12, 14, 16), max_fp=0, max_fp_rate_hi=0.05, margin_steps=1)

VARIANTS: dict[str, ProbSpec] = {
    "A_em": SPEC,
    "B_labels": replace(SPEC, model=replace(SPEC.model, m_method="labels")),
    "C_no_term_frequency": replace(SPEC, model=without_tf(SPEC.model)),
    "D_exact_keys_only": replace(SPEC, blockers=tuple(b for b in GENERATORS if isinstance(b, KeyBlocker))),
    "E_levenshtein_names": replace(SPEC, model=with_level(with_level(SPEC.model, "name", "fold JW high", function="levenshtein", threshold=1),
                                                          "name", "fold JW low", function="levenshtein", threshold=2)),
    "F_constrained_given_name": replace(SPEC, resolver=Resolver(method="constrained", cannot_link=("GIVEN_NAME_KANA",))),
}

# Decision rule, applied in order: each change is applied ON TOP OF the configuration chosen so far, and adopted if
#   "better":  a significant paired gain on cross-validated train (p < DECISION_P, no FP added)
#   "cheaper": no paired loss on cross-validated train (no TP lost, no FP added); it costs less
# and in both cases no paired loss on any robustness dataset.
DECISIONS = (
    ("no_term_frequency", "better", lambda s: replace(s, model=without_tf(s.model))),
    ("exact_keys_only", "cheaper", lambda s: replace(s, blockers=tuple(b for b in s.blockers if isinstance(b, KeyBlocker)))),
)
DECISION_P = 0.05

PTUNE = ProbTune(
    toggle_blockers=("person_name", "company_name", "sn_person_name", "sn_company_name", "tfidf_person_name", "tfidf_company_name"),
    m_methods=("em", "labels"),
    tf_options=(True, False),
    level_thresholds={("name", "fold JW high"): (0.88, 0.97, 0.01), ("name", "fold JW low"): (0.75, 0.90, 0.01),
                      ("company", "JW high"): (0.88, 0.97, 0.01), ("company", "JW low"): (0.75, 0.90, 0.01)},
    weight_range=(0.0, 16.0, 0.5),
    resolvers={"connected_components": Resolver(method="connected_components"),
               "constrained_given_name": Resolver(method="constrained", cannot_link=("GIVEN_NAME_KANA",))},
    n_trials=30,
    timeout_s=1800,
    seed=0,
    storage_path=Path("../data/artifacts/optuna/probabilistic_matching.db"),
    study_prefix="probabilistic",
    dashboard_host="127.0.0.1",
    dashboard_port=8081,  # 8080 belongs to the deterministic notebook's dashboard
)

## Setup
Three caches. None of them changes a result; they only skip repeated work.
- **Models** are cached by (config, fitting set), so variants that share a model train it once. **Final models** fit on all of train; **cross-validation models** fit on k−1 train folds.
- **u** (≈97% of training time) depends only on the comparisons and the fitting records (`FSModel.u_signature`). A model that differs only in m method or term-frequency adjustment reuses it, which gives an identical model (self-checked in the Training cell).
- **Runs** (and cross-validation runs) are cached by (config, dataset, threshold). The Decision, Robustness and Output cells repeat many of the same runs.

In [ ]:
CON, DB = connect()
self_check(STANDARDIZE)
uid = DATA.unique_id
l, r = f"{uid}_l", f"{uid}_r"
train_records, train_truth = load(DATA, "train")
train_prev_pairs, train_prev_resolved = read_stage(DATA, "train", SPEC.previous_stage)
FOLDS = entity_folds(train_truth, DATA, CV_FOLDS, CV_SEED)
MODELS: dict[tuple[str, object], dict] = {}
U_CACHE: dict[tuple[str, object], dict] = {}
RUNS: dict[tuple[str, str, float], ProbResult] = {}
CVS: dict[tuple[str, float], list[ProbResult]] = {}


def fit(spec: ProbSpec, records: pd.DataFrame, truth: pd.DataFrame, key: object) -> dict:
    if (repr(spec.model), key) not in MODELS:
        df = prepare(spec.standardize, records, spec.normalized(), spec.used())
        u = U_CACHE.get((spec.model.u_signature(), key))  # same comparisons + same records → same u
        model_json = train_fs(spec.model, df, DB, uid, labels=truth.set_index(uid)[DATA.entity_id], u=u)
        U_CACHE.setdefault((spec.model.u_signature(), key), u_values(model_json))
        MODELS[repr(spec.model), key] = model_json
    return MODELS[repr(spec.model), key]


def model_for(spec: ProbSpec) -> dict:  # fitted on all of train
    return fit(spec, train_records, train_truth, "train")


def run(spec: ProbSpec, dataset: str, min_match_weight: float | None = None) -> ProbResult:
    key = (repr(spec), dataset, spec.min_match_weight if min_match_weight is None else min_match_weight)
    if key not in RUNS:
        records, truth = load(DATA, dataset)
        prev_pairs, prev_resolved = read_stage(DATA, dataset, spec.previous_stage)
        RUNS[key] = run_fs(spec, model_for(spec), records, truth, prev_pairs, prev_resolved, DB, DATA, min_match_weight)
    return RUNS[key]


def cv(spec: ProbSpec, min_match_weight: float | None = None) -> list[ProbResult]:  # out-of-fold results on train
    key = (repr(spec), spec.min_match_weight if min_match_weight is None else min_match_weight)
    if key not in CVS:
        CVS[key] = cross_validate_fs(spec, fit, train_records, train_truth, train_prev_pairs, train_prev_resolved, FOLDS,
                                     DB, DATA, min_match_weight)
    return CVS[key]


def choose_threshold(spec: ProbSpec) -> tuple[float, pd.DataFrame]:
    results = cv(spec)
    curve = cv_threshold_curve(results, spec, train_prev_pairs, train_truth, FOLDS, DB, DATA, list(THRESHOLD_RULE.grid))
    return select_threshold(curve, THRESHOLD_RULE)


def joined(results: ProbResult | list[ProbResult]) -> set[tuple[str, str]]:  # every pair the clusters put together
    results = results if isinstance(results, list) else [results]
    return set().union(*(predicted_pairs(res.records.set_index(uid)["CLUSTER_ID"]) for res in results))


def truth_set(truth: pd.DataFrame) -> set[tuple[str, str]]:
    return set(map(tuple, true_pairs(truth.set_index(uid)[DATA.entity_id], uid).to_numpy()))

## Design analysis on train only
Which keys reach the true pairs the deterministic stage **missed**? This justifies the candidate keys. It runs on **train** only, so the design can't leak val or test.

In [ ]:
t_df = prepare(SPEC.standardize, train_records, SPEC.normalized(), SPEC.used())
t_entity = train_truth.set_index(uid)[DATA.entity_id]
t_prev = train_prev_resolved.set_index(uid)["cluster_id"].reindex(t_df[uid].to_numpy())
targets = missed_pairs(t_prev, t_entity.reindex(t_prev.index), uid)
side = t_df.set_index(uid)
with_entity = t_df.assign(__entity=t_df[uid].map(t_entity))
rows = []
for b in GENERATORS:
    if isinstance(b, KeyBlocker):
        cols = [k.name for k in b.keys]
        left, right = side.loc[targets[l], cols].reset_index(drop=True), side.loc[targets[r], cols].reset_index(drop=True)
        agree = (left == right).fillna(False).all(axis=1)  # NA never counts as agreement
        keyed = with_entity.dropna(subset=cols)
        rows.append(dict(key=b.name, missed_pairs_agree=float(agree.mean()),
                         different_entity_pairs=c2(keyed.groupby(cols).size()) - c2(keyed.groupby(cols + ["__entity"]).size())))
print(f"train: {len(targets):,} true pairs missed by the deterministic stage · cost O(N) per key")
display(pd.DataFrame(rows).round(4))

## Train Fellegi–Sunter on train: EM vs labels
- **λ** (the prior) comes from near-certain rules and their assumed recall.
- **u** comes from random pairs.
- **m** comes from **EM** (unsupervised; works on real data) or from **labels** (train truth; the supervised upper bound).

The weight of a level is log2(m/u). The self-check confirms that scoring a registered candidate table gives exactly the same weights as Splink's own blocking.

In [ ]:
n, n_comp = len(t_df), len(SPEC.model.comparisons)
prior_p = sum(estimate(KeyBlocker(name="prior", keys=rule), t_df, n)["est_pairs"] for rule in SPEC.model.prior_rules)
em_p = sum(estimate(KeyBlocker(name="em", keys=rule), t_df, n)["est_pairs"] for rule in SPEC.model.em_rules)
print(f"training cost: λ O(N + P_rules) = O({n:,} + {prior_p:,}) · u O(max_pairs·C) = O({SPEC.model.u_max_pairs:,.0f}·{n_comp}) · "
      f"m by EM O(iterations·P_em·C) = O(it·{em_p:,}·{n_comp}); by labels O(Σ k² over entities)")
DATA.models_dir.mkdir(parents=True, exist_ok=True)
weights = {}
for method in ("em", "labels"):
    spec = replace(SPEC, model=replace(SPEC.model, m_method=method))
    t = time.perf_counter()
    model_json = model_for(spec)
    (DATA.models_dir / f"{spec.model.name}_{method}.json").write_text(json.dumps(model_json, ensure_ascii=False, indent=1))
    weights[method] = parameters(model_json).set_index(["comparison", "level"])
    print(f"{method}: trained in {time.perf_counter() - t:.1f}s, λ = {model_json['probability_two_random_records_match']:.2e}")
display(pd.concat({k: v[["m", "u", "weight"]] for k, v in weights.items()}, axis=1).round(3))

# self-check: the labels model reused the EM model's u (same u_signature); retraining it from scratch must give the same model
labels_spec = replace(SPEC, model=replace(SPEC.model, m_method="labels"))
fresh = train_fs(labels_spec.model, t_df, DB, uid, labels=train_truth.set_index(uid)[DATA.entity_id])
assert parameters(fresh).fillna(0).round(12).equals(parameters(model_for(labels_spec)).fillna(0).round(12))
print("self-check: reusing u gives the same model as estimating it again")

val_df = prepare(SPEC.standardize, load(DATA, "val")[0], SPEC.normalized(), SPEC.used())
model_json = model_for(SPEC)
native = Linker(DB.register(val_df), {**model_json, "blocking_rules_to_generate_predictions": [keys_sql(SPEC.model.em_rules[0])]},
                log_level=logging.WARNING).inference.predict().as_pandas_dataframe().astype({l: "string", r: "string"})
ours = score_fs(model_json, val_df, native[[l, r]], DB, uid)
both = native.merge(ours, on=[l, r], suffixes=("_native", "_registered"))
assert len(both) == len(native) and np.allclose(both["match_weight_native"], both["match_weight_registered"], atol=1e-9)
print(f"self-check: {len(both):,} pairs score identically through a registered candidate table")

## Candidates on val: what each generator finds, and at what cost
Only pairs touching a record the deterministic stage left unresolved are generated. `completeness_on_targets` is the share of the true pairs the deterministic stage missed that the candidates contain, i.e. the ceiling for this stage.

In [ ]:
val_res = run(replace(SPEC, blockers=GENERATORS), "val")
display(val_res.report.drop(columns=["true_in_candidates"]).round(4))
val_records, val_truth = load(DATA, "val")
entity = val_truth.set_index(uid)[DATA.entity_id]
prev = read_stage(DATA, "val", SPEC.previous_stage)[1].set_index(uid)["cluster_id"]
val_targets = missed_pairs(prev, entity.reindex(prev.index), uid)
found_by, cumulative = val_res.candidates["sources"].str.split("+"), []
for i, b in enumerate(GENERATORS):
    names = {x.name for x in GENERATORS[: i + 1]}
    m = candidate_metrics(val_res.candidates[found_by.map(lambda s: bool(names & set(s)))], entity, uid, val_targets)
    cumulative.append(dict(up_to=b.name, candidates=m["candidates"], completeness_on_targets=m["completeness_on_targets"]))
display(pd.DataFrame(cumulative).round(4))

## Threshold: chosen on cross-validated train, confirmed on val
The rule (`THRESHOLD_RULE`): the lowest weight with at most `max_fp` false accepted pairs and a stage FP-rate upper bound ≤ `max_fp_rate_hi`, on **out-of-fold** train scores, then `margin_steps` stricter. Val is not used to choose, only to confirm.

In [ ]:
t = time.perf_counter()
weight, cv_curve = choose_threshold(SPEC)
print(f"cross-validation: {CV_FOLDS} folds · O(k · (fit + candidates + score + resolve)) · {time.perf_counter() - t:.1f}s")
display(cv_curve[["min_match_weight", "probability", "accepted", "pairs_FP", "stage_fp_rate_hi", "TP", "FP", "recall",
                  "recall_lo", "recall_hi", "fp_rate_hi", "eligible"]].round(4))
SPEC = replace(SPEC, min_match_weight=weight)
VARIANTS = {k: replace(v, min_match_weight=weight) for k, v in VARIANTS.items()}
print(f"rule → min_match_weight = {weight} (probability {1 / (1 + 2 ** -weight):.4f})")
val_curve = threshold_curve(run(SPEC, "val"), SPEC, read_stage(DATA, "val", SPEC.previous_stage)[0], val_truth, DB, DATA,
                            list(THRESHOLD_RULE.grid))
print("val (confirmation only):")
display(val_curve[["min_match_weight", "accepted", "pairs_FP", "TP", "FP", "recall", "recall_lo", "recall_hi", "fp_rate_hi"]].round(4))

## Evaluation on val
- **Cascade metrics:** the `deterministic` row reproduces the previous stage; the `probabilistic` row adds this stage.
- **Column meaning:** `pairs_*` = matching; the rest = resolution.

In [ ]:
val_res = run(SPEC, "val")
display(val_res.metrics.round(4))
print({k: round(v, 4) if isinstance(v, float) else v for k, v in val_res.stage_metrics.items()})
cols = sorted(c for c in SPEC.used() if c in val_res.records)
side = val_res.records.set_index(uid)[cols]
accepted = val_res.scored[val_res.scored["match_weight"] >= SPEC.min_match_weight]
fp = accepted[accepted["fp"]].head(DATA.show_rows)
print(f"val false-positive pairs accepted at weight ≥ {SPEC.min_match_weight}: {int(accepted['fp'].sum())}")
if len(fp):
    display(pd.concat([fp[["match_weight", "sources"]].reset_index(drop=True),
                       side.loc[fp[l]].add_suffix("_l").reset_index(drop=True),
                       side.loc[fp[r]].add_suffix("_r").reset_index(drop=True)], axis=1).T)
near = val_res.scored[~val_res.scored["fp"] & (val_res.scored["match_weight"] < SPEC.min_match_weight)].nlargest(DATA.show_rows, "match_weight")
print("true pairs just below the threshold (what a lower threshold would add):")
display(pd.concat([near[["match_weight", "sources"]].reset_index(drop=True),
                   side.loc[near[l]].add_suffix("_l").reset_index(drop=True),
                   side.loc[near[r]].add_suffix("_r").reset_index(drop=True)], axis=1).T)

## A/B: variants on val, paired against SPEC
- **`TP_gains` / `TP_losses`:** true pairs only the variant joins / only `SPEC` joins
- **`p`:** an exact sign test on those counts
- **`FP_added` / `FP_removed`:** the same for false pairs

Rows are appended to the leaderboard.

In [ ]:
val_true = truth_set(val_truth)
base = joined(run(SPEC, "val"))
ab = []
for name, spec in VARIANTS.items():
    t = time.perf_counter()
    res = run(spec, "val")
    m, s = res.metrics.iloc[-1], res.stage_metrics
    row = dict(FP=m["FP"], fp_rate_hi=m["fp_rate_hi"], recall=m["recall"], recall_lo=m["recall_lo"], recall_hi=m["recall_hi"],
               precision=m["precision"], f1=m["f1"], overmerged=m["overmerged"], **paired_test(base, joined(res), val_true),
               candidates=s["candidates"], completeness_on_targets=s["completeness_on_targets"], accepted=s["pairs"],
               average_precision=s["average_precision"], seconds=round(time.perf_counter() - t, 1))
    append_leaderboard(DATA, f"probabilistic:{name}", spec, "val", row)
    ab.append(dict(variant=name, **row))
display(pd.DataFrame(ab).round(4))

## Decision: adopt a variant only by rule
Each entry in `DECISIONS` is compared with the configuration chosen so far, **paired**:
- on **cross-validated train** (out-of-fold, at the chosen threshold)
- on every **robustness** dataset

It is adopted only if it passes the rule written in the Configuration cell. Val and test play no part, so the val A/B above stays an honest check.

In [ ]:
train_true = truth_set(train_truth)
chosen, verdicts = SPEC, []
for name, kind, change in DECISIONS:
    variant = change(chosen)
    t = time.perf_counter()
    cv_test = paired_test(joined(cv(chosen, chosen.min_match_weight)), joined(cv(variant, variant.min_match_weight)), train_true)
    robust = {d: paired_test(joined(run(chosen, d)), joined(run(variant, d)), truth_set(load(DATA, d)[1])) for d in DATA.robustness}
    no_loss_robust = all(x["TP_gains"] >= x["TP_losses"] and x["FP_added"] <= x["FP_removed"] for x in robust.values())
    if kind == "better":
        ok = cv_test["TP_gains"] > cv_test["TP_losses"] and cv_test["p"] < DECISION_P and cv_test["FP_added"] == 0 and no_loss_robust
    else:
        ok = cv_test["TP_losses"] == 0 and cv_test["FP_added"] == 0 and no_loss_robust
    verdicts.append(dict(variant=name, rule=kind, adopted=ok, **{f"cv_{k}": v for k, v in cv_test.items()},
                         **{f"{d}_{k}": v for d, x in robust.items() for k, v in x.items() if k in ("TP_gains", "TP_losses", "FP_added")},
                         seconds=round(time.perf_counter() - t, 1)))
    if ok:
        chosen = variant
display(pd.DataFrame(verdicts).round(4))
if chosen is not SPEC:
    weight, cv_curve = choose_threshold(chosen)  # the threshold rule again, for the adopted configuration
    chosen = replace(chosen, min_match_weight=weight)
    print(f"adopted; threshold re-chosen on cross-validated train → {weight}")
SPEC = chosen
print(f"SPEC: {len(SPEC.blockers)} generators · term frequency {'on' if any(c.tf for c in SPEC.model.comparisons) else 'off'} · "
      f"m by {SPEC.model.m_method} · min_match_weight {SPEC.min_match_weight} · resolver {SPEC.resolver.method}")

## Robustness: SPEC on stress and large (report only)
- **stress** (another seed, more typos, less nesting, more households): exposes overfitting to the generator.
- **large** (50k entities): its many predicted pairs certify a tight FP-rate upper bound.

In [ ]:
robust = {d: run(SPEC, d) for d in DATA.robustness}
display(pd.concat({d: r.metrics for d, r in robust.items()}, names=["dataset"])[
    ["stage", "TP", "FP", "recall", "recall_lo", "recall_hi", "fp_rate_hi", "overmerged", "dup_unresolved"]].round(4))
for d, res in robust.items():
    m, s = res.metrics.iloc[-1], res.stage_metrics
    print(f"{d}: {res.report['big_o'].unique().tolist()} · {s['candidates']:,} candidates · {res.seconds:.1f}s · "
          f"stage: {s['pairs_FP']} FP in {s['pairs']:,} accepted pairs → FP rate ≤ {s['pairs_fp_rate_hi']:.3%} · "
          f"overall: {int(m['FP'])} FP in {int(m['TP'] + m['FP']):,} predicted pairs → ≤ {m['fp_rate_hi']:.3%}")

# error analysis only (nothing here feeds back into SPEC): what do the false positives at scale look like?
res = robust[DATA.robustness[-1]]
bad = res.scored[(res.scored["match_weight"] >= SPEC.min_match_weight) & res.scored["fp"]].nlargest(DATA.show_rows, "match_weight")
side = res.records.set_index(uid)[sorted(c for c in SPEC.used() if c in res.records)]
print(f"{DATA.robustness[-1]}: highest-weight false-positive pairs")
if len(bad):
    display(pd.concat([bad[["match_weight", "sources"]].reset_index(drop=True),
                       side.loc[bad[l]].add_suffix("_l").reset_index(drop=True),
                       side.loc[bad[r]].add_suffix("_r").reset_index(drop=True)], axis=1).T)

## Output: SPEC on every dataset → standard files and gold (test = the final report)
- `data/artifacts/matched/<dataset>/probabilistic_pairs.csv`: accepted pairs
- `data/artifacts/resolved/<dataset>/probabilistic_resolved.csv`: deterministic + probabilistic clusters
- `data/artifacts/unresolved/<dataset>/probabilistic_unresolved.csv`: the hand-off to a next stage
- `data/medallion/gold/<dataset>.csv`: resolved entities (CONT_ID → cluster), the gold layer

Truth-derived columns are never written. The test counter flags repeated test evaluations.

In [ ]:
for dataset in DATA.datasets:
    res = run(SPEC, dataset)
    records = load(DATA, dataset)[0]
    acc = res.scored[res.scored["match_weight"] >= SPEC.min_match_weight].assign(stage=SPEC.stage, rule=SPEC.model.name)
    pairs = acc[[l, r, "stage", "rule", "match_weight", "match_probability", "sources"]].assign(score=acc["match_probability"], kept=True)
    resolved = res.records[[uid, "CLUSTER_ID", "MATCH_STAGE", "MATCH_RULE"]].set_axis([uid, "cluster_id", "resolved_stage", "resolved_by"], axis=1)
    unresolved = res.records.loc[res.records["MATCH_STAGE"].isna(), list(records.columns)]
    assert len(unresolved) + resolved["resolved_stage"].notna().sum() == len(records)
    write_stage(DATA, dataset, SPEC.stage, pairs, resolved, unresolved)
    write_gold(DATA, dataset, resolved)
    m = res.metrics.iloc[-1]
    append_leaderboard(DATA, "probabilistic:SPEC", SPEC, dataset,
                       {**m.drop("stage").to_dict(), **{f"stage_{k}": v for k, v in res.stage_metrics.items()}})
    print(f"{dataset:>6}: recall {res.metrics.iloc[0]['recall']:.4f} → {m['recall']:.4f} [{m['recall_lo']:.4f}, {m['recall_hi']:.4f}] · "
          f"FP {m['FP']} (rate ≤ {m['fp_rate_hi']:.3%}) · {len(acc):,} accepted · {len(unresolved):,} unresolved · {res.seconds:.1f}s")
counter = test_evaluations(DATA)
display(counter)
for _, row in counter[counter["configs_on_test"] > 1].iterrows():
    print(f"warning: {row['configs_on_test']} configurations of {row['family']!r} were scored on test; its test numbers are no longer unbiased")

## Optuna: search component combinations with a cross-validated objective
Each trial picks:
- which generators to switch off
- m method, term frequency on/off
- level thresholds, `min_match_weight`, the resolver

The objectives (**minimise FP**, **maximise recall**) are computed **out-of-fold** on train: k folds, each scored by a model fitted on the other folds. Tuning therefore never rewards fitting noise in the scored records. Models are cached by (config, fold).

The Pareto front is re-scored on **val** with intervals. The **suggested** point is the cheapest configuration within the noise of the best 0-FP val recall (`pick_within_noise`).

Dashboard: `http://127.0.0.1:8081`. From a terminal in `notebooks/`: `../.venv/bin/optuna-dashboard --port 8081 sqlite:///../data/artifacts/optuna/probabilistic_matching.db`.

In [ ]:
import optuna

from er.tune import new_study, pick_within_noise, plain, start_dashboard


def spec_for(trial: optuna.Trial | optuna.trial.FixedTrial) -> ProbSpec:
    off = {b for b in PTUNE.toggle_blockers if not trial.suggest_categorical(f"use:{b}", [True, False])}
    model = replace(SPEC.model, m_method=trial.suggest_categorical("m_method", list(PTUNE.m_methods)))
    if not trial.suggest_categorical("term_frequency", list(PTUNE.tf_options)):
        model = without_tf(model)
    for (comparison, label), (lo, hi, step) in PTUNE.level_thresholds.items():
        model = with_level(model, comparison, label, threshold=trial.suggest_float(f"{comparison}|{label}", lo, hi, step=step))
    lo, hi, step = PTUNE.weight_range
    return replace(SPEC, blockers=tuple(b for b in GENERATORS if b.name not in off), model=model,
                   min_match_weight=trial.suggest_float("min_match_weight", lo, hi, step=step),
                   resolver=PTUNE.resolvers[trial.suggest_categorical("resolver", list(PTUNE.resolvers))])


def objective(trial: optuna.Trial) -> tuple[float, float]:
    spec = spec_for(trial)
    try:
        s = summed(cv(spec, spec.min_match_weight))
    except OverBudget as e:
        raise optuna.TrialPruned(str(e))
    trial.set_user_attr("big_o", f"O(k · (fit + candidates + score + resolve)), k = {CV_FOLDS}")
    for k in ("recall_lo", "recall_hi", "fp_rate_hi", "accepted", "accepted_FP", "candidates", "precision"):
        trial.set_user_attr(k, plain(s[k]))
    return float(s["FP"]), float(s["recall"])


storage = start_dashboard(PTUNE)
t = time.perf_counter()
summed(cv(SPEC, SPEC.min_match_weight))
per_trial = time.perf_counter() - t
print(f"study: O(T · k · (fit + candidates + score + resolve)) = O({PTUNE.n_trials} · {CV_FOLDS} · …); one cross-validated trial ≈ "
      f"{per_trial:.1f}s (cached models) to a few times that when the model changes → T = {PTUNE.n_trials} (timeout {PTUNE.timeout_s / 60:.0f} min)")
study = new_study(PTUNE, storage, ["FP", "recall"], ["minimize", "maximize"])
study.optimize(objective, n_trials=PTUNE.n_trials, timeout=PTUNE.timeout_s, show_progress_bar=True)
trials = study.trials_dataframe(attrs=("number", "values", "params", "user_attrs", "state"))
display(trials.rename(columns={"values_FP": "FP", "values_recall": "recall"}).sort_values(["FP", "recall"], ascending=[True, False]).head(15))

In [ ]:
front, seen = [], set()
for t in sorted(study.best_trials, key=lambda t: (t.values[0], -t.values[1])):
    spec = spec_for(optuna.trial.FixedTrial(t.params))
    if repr(spec) in seen:
        continue
    seen.add(repr(spec))
    v = run(spec, "val").metrics.iloc[-1]
    front.append(dict(trial=t.number, cv_FP=t.values[0], cv_recall=t.values[1], val_FP=v["FP"], val_recall=v["recall"],
                      val_recall_lo=v["recall_lo"], val_recall_hi=v["recall_hi"], val_fp_rate_hi=v["fp_rate_hi"], val_zero_fp=v["FP"] == 0,
                      candidates=t.user_attrs["candidates"], m_method=t.params["m_method"], term_frequency=t.params["term_frequency"],
                      min_match_weight=t.params["min_match_weight"], resolver=t.params["resolver"],
                      generators_off=",".join(b for b in PTUNE.toggle_blockers if not t.params[f"use:{b}"]) or "—"))
front = pd.DataFrame(front)
print(f"Pareto front: {len(front)} distinct configurations ({len(study.best_trials)} non-dominated trials), cross-validated train vs val")
display(front.round(4))
if len(front):
    suggested = pick_within_noise(front, "val_recall", "val_recall_lo", "val_recall_hi", "val_zero_fp", "candidates")
    print(f"suggested (cheapest within the noise of the best 0-FP val recall): trial {front.at[suggested, 'trial']}")